In [57]:
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [58]:
data = pd.read_csv('../datasets/car_insurance_fraud.CSV')
y = data['fraud_reported'].replace({'Y': 1, 'N': 0}).astype(int)
X = data.drop(['fraud_reported'], axis=1)

# Turn date columns into numeric features.
policy_bind_date = pd.to_datetime(X['policy_bind_date'], dayfirst=True, errors='coerce')
incident_date = pd.to_datetime(X['incident_date'], dayfirst=True, errors='coerce')
X['policy_bind_year'] = policy_bind_date.dt.year
X['policy_bind_month'] = policy_bind_date.dt.month
X['policy_bind_day'] = policy_bind_date.dt.day
X['incident_year'] = incident_date.dt.year
X['incident_month'] = incident_date.dt.month
X['incident_day'] = incident_date.dt.day
X['incident_dayofweek'] = incident_date.dt.dayofweek
X['days_between_bind_and_incident'] = (incident_date - policy_bind_date).dt.days

# Split composite coverage into two numeric columns.
policy_csl = X['policy_csl'].str.split('/', expand=True)
X['policy_csl_liability'] = pd.to_numeric(policy_csl[0], errors='coerce')
X['policy_csl_property'] = pd.to_numeric(policy_csl[1], errors='coerce')

# Encode binary and ordinal fields explicitly.
X['police_report_available'] = X['police_report_available'].replace({'YES': 1, 'NO': 0, '?': -1}).astype(int)
X['property_damage'] = X['property_damage'].replace({'YES': 1, 'NO': 0, '?': -1}).astype(int)
X['incident_severity'] = X['incident_severity'].replace({
    'Trivial Damage': 0,
    'Minor Damage': 1,
    'Major Damage': 2,
    'Total Loss': 3,
}).astype(int)

# Drop identifiers and high-cardinality text that are unlikely to help the model.
X = X.drop(columns=['policy_bind_date', 'incident_date', 'policy_csl', 'policy_number', 'insured_zip', 'incident_location'])

# One-hot encode the remaining nominal categories.
X = pd.get_dummies(
    X,
    columns=[
        'policy_state',
        'insured_sex',
        'insured_education_level',
        'insured_occupation',
        'insured_hobbies',
        'insured_relationship',
        'incident_type',
        'collision_type',
        'authorities_contacted',
        'incident_state',
        'incident_city',
        'auto_make',
        'auto_model',
    ],
    dtype=int,
)

data

C:\Users\Anna\AppData\Local\Temp\ipykernel_22400\420200538.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = data['fraud_reported'].replace({'Y': 1, 'N': 0}).astype(int)
C:\Users\Anna\AppData\Local\Temp\ipykernel_22400\420200538.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X['police_report_available'] = X['police_report_available'].replace({'YES': 1, 'NO': 0, '?': -1}).astype(int)
C:\Users\Anna\AppData\Local\Temp\ipykernel_22400\420200538.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will 

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,521585,17.10.2014,OH,250/500,1000,1406.91,0,466132,...,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y
1,228,42,342868,27.06.2006,IN,250/500,2000,1197.22,5000000,468176,...,0,?,5070,780,780,3510,Mercedes,E400,2007,Y
2,134,29,687698,06.09.2000,OH,100/300,2000,1413.14,5000000,430632,...,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N
3,256,41,227811,25.05.1990,IL,250/500,2000,1415.74,6000000,608117,...,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y
4,228,44,367455,06.06.2014,IL,500/1000,1000,1583.91,6000000,610706,...,1,NO,6500,1300,650,4550,Accura,RSX,2009,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,3,38,941851,16.07.1991,OH,500/1000,1000,1310.80,0,431289,...,1,?,87200,17440,8720,61040,Honda,Accord,2006,N
996,285,41,186934,05.01.2014,IL,100/300,1000,1436.79,0,608177,...,3,?,108480,18080,18080,72320,Volkswagen,Passat,2015,N
997,130,34,918516,17.02.2003,OH,250/500,500,1383.49,3000000,442797,...,3,YES,67500,7500,7500,52500,Suburu,Impreza,1996,N
998,458,62,533940,18.11.2011,IL,500/1000,2000,1356.92,5000000,441714,...,1,YES,46980,5220,5220,36540,Audi,A5,1998,N


In [66]:
X = X.select_dtypes(include=[np.number]) #filter X to only include numeric columns
X = X.fillna(X.mean()) #fill NaN values with the mean of the column


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_train_means = X_train.mean().to_frame().T

COLUMNS = X_train.columns

nn = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        solver="adam",
        activation="tanh",
        hidden_layer_sizes=(1024, 512, 256),
        alpha=1e-5,
        learning_rate_init=3e-4,
        batch_size=32,
        early_stopping=False,
        max_iter=10000,
        random_state=0,
    ),
)
nn.fit(X_train, y_train)

def print_accuracy(f):
    print(
        "Accuracy = {0}%".format(
            100 *
            np.mean(
                f(X_test) == y_test))
    )
    time.sleep(0.5)  # to let the print get out before any progress bars

print_accuracy(nn.predict)

Accuracy = 78.0%


In [ ]:
import pickle

with open('../datasets/car_insurance_fraud_nn.pkl', 'wb') as file:

    # A new file will be created
    pickle.dump(nn, file)

In [ ]:
with open('../datasets/car_insurance_fraud_train.csv', 'wb') as file:
    
        # A new file will be created
        X_train.to_csv(file, index=False)

In [ ]:
with open('../datasets/car_insurance_fraud_traintruth.csv', 'wb') as file:
    
    y_train.to_csv(file, index=False)

In [ ]:
with open('../datasets/car_insurance_fraud_predict.csv', 'wb') as file:

    predictions = pd.DataFrame(nn.predict_proba(X_train), columns=[ 'Delete','Fraud_prediction'])
    predictions = predictions['Fraud_prediction']

    full = pd.concat([X_train.reset_index(drop=True), predictions], axis=1)

    full.to_csv(file, index=False)